# Cuadro mágico vs. matriz de riesgo

**Protocolos de Comunicación y Seguridad Informática · TEINCO**
Neider Steven Peña Riaño · basado en el informe de riesgos ISO 31000 con modelo RAV

## Cómo usarlo

1. Dale **▶** a la celda **«Cuadro mágico vs. matriz de riesgo»**. La herramienta aparece debajo.
2. Usa las pestañas: **Vista general**, **Paso a paso** y **Escenarios RAV**.
3. Para exponer, pulsa **Modo sustentación**: son 6 pantallas en orden. Lo que conviene decir en cada una está en la guía de estudio, al final del cuaderno.
4. Si quieres comprobar los cálculos frente al profe, dale **▶** a la celda **«Pruebas automáticas»**.

> **Idea central:** Python hace todos los cálculos y los verifica. La página web solo muestra los resultados.

Si algo se ve raro, vuelve a darle ▶ a la celda principal y todo queda como al inicio.
Para tener un respaldo sin internet, marca la casilla **descargar_respaldo** y dale ▶: se descarga un `.html` que abre con doble clic.


In [ ]:
#@title ▶ Cuadro mágico vs. matriz de riesgo { display-mode: "form" }
#@markdown Dale ▶ a esta celda y la herramienta aparece debajo. No hay que tocar el código.
#@markdown
#@markdown Marca la casilla si quieres descargar el respaldo `.html` (abre sin Colab ni internet).
descargar_respaldo = False  #@param {type:"boolean"}

# =============================================================================
#  CUADRO MÁGICO VS. MATRIZ DE RIESGO
#  Protocolos de Comunicación y Seguridad Informática · TEINCO
#  Autor: Neider Steven Peña Riaño
#
#  Idea central del programa:
#    - Python hace TODOS los cálculos (capas 1, 2 y 3).
#    - La interfaz (capa 4) es una página web que solo MUESTRA lo ya calculado.
# =============================================================================

import json
import numpy as np
from IPython.display import HTML, display


# -----------------------------------------------------------------------------
# CAPA 1 · DATOS DEL INFORME
# Todo sale del informe ISO 31000. Son fijos: la interfaz no los puede cambiar.
# -----------------------------------------------------------------------------

# Escala de frecuencia (filas de la matriz): (nombre, valor)
FRECUENCIA = [("Rara vez", 1), ("Eventual", 2), ("Ocasional", 3),
              ("Frecuente", 4), ("Muy frecuente", 5)]

# Escala de impacto (columnas de la matriz): (nombre, valor).
# Crece mucho más rápido que la frecuencia: lo grave pesa más.
IMPACTO = [("Inferior", 1), ("Menor", 3), ("Importante", 10),
           ("Mayor", 25), ("Superior", 50)]

# Niveles de aceptación: (nombre, tope en % del riesgo máximo). None = sin tope.
NIVELES = [("Bajo", 2), ("Mediano", 5), ("Alto", 20), ("Extremo", None)]

# Factores del modelo RAV (Tabla 3 del informe): categoría -> criterio -> factor
FACTORES_RAV = {
    "Desconocido":    {"Percibido": 1.0, "Identificado": 0.1, "Verificado": 0.2, "No aplicado": 0.0},
    "Filtrado":       {"Percibido": 1.0, "Identificado": 0.2, "Verificado": 0.4, "No aplicado": 0.1},
    "Preocupación":   {"Percibido": 1.0, "Identificado": 0.4, "Verificado": 0.8, "No aplicado": 0.2},
    "Debilidad":      {"Percibido": 1.0, "Identificado": 0.8, "Verificado": 1.6, "No aplicado": 0.3},
    "Vulnerabilidad": {"Percibido": 1.0, "Identificado": 1.6, "Verificado": 3.2, "No aplicado": 0.4},
}

# Los 15 escenarios del informe: (ID, situación, frecuencia, impacto, criterio, categoría RAV)
ESCENARIOS = [
    ("R01", "Denegación de servicio (DoS) al servidor CRM", "Ocasional", "Importante", "Identificado", "Preocupación"),
    ("R02", "Falla del aire acondicionado de los racks del datacenter", "Frecuente", "Superior", "Verificado", "Vulnerabilidad"),
    ("R03", "Carencia de backups de los sistemas de misión crítica", "Muy frecuente", "Mayor", "Verificado", "Debilidad"),
    ("R04", "Ransomware que ingresa por correo de phishing", "Eventual", "Superior", "Identificado", "Vulnerabilidad"),
    ("R05", "Contraseñas compartidas entre operadores de turno", "Muy frecuente", "Menor", "Verificado", "Debilidad"),
    ("R06", "Parches de seguridad atrasados en servidores Windows", "Frecuente", "Importante", "Identificado", "Debilidad"),
    ("R07", "Corte de energía sin UPS en la sala de servidores", "Rara vez", "Mayor", "Percibido", "Preocupación"),
    ("R08", "Fuga de información por memorias USB no controladas", "Ocasional", "Mayor", "Percibido", "Desconocido"),
    ("R09", "Error humano en la configuración del firewall perimetral", "Eventual", "Importante", "Identificado", "Filtrado"),
    ("R10", "Robo de portátil corporativo sin cifrado de disco", "Rara vez", "Importante", "No aplicado", "Filtrado"),
    ("R11", "Caída del enlace principal de internet", "Frecuente", "Menor", "Verificado", "Preocupación"),
    ("R12", "Cuentas de exempleados que siguen activas", "Ocasional", "Mayor", "Verificado", "Vulnerabilidad"),
    ("R13", "Spam masivo en los buzones corporativos", "Muy frecuente", "Inferior", "No aplicado", "Filtrado"),
    ("R14", "Inundación en el cuarto de comunicaciones", "Rara vez", "Superior", "Percibido", "Desconocido"),
    ("R15", "Inyección SQL en el portal web de clientes", "Eventual", "Mayor", "Identificado", "Vulnerabilidad"),
]

# Opciones permitidas para el cuadro mágico
TAMANOS = (3, 5, 7, 9)
LADOS = ("arriba", "abajo", "izquierda", "derecha")

REGLAS_LADO = {
    "arriba":    "Empieza en el centro de arriba, avanza en diagonal arriba-derecha y, si la casilla está ocupada, baja una.",
    "abajo":     "Empieza en el centro de abajo, avanza en diagonal abajo-derecha y, si la casilla está ocupada, sube una.",
    "izquierda": "Empieza en el centro de la izquierda, avanza en diagonal abajo-izquierda y, si la casilla está ocupada, se corre una a la derecha.",
    "derecha":   "Empieza en el centro de la derecha, avanza en diagonal abajo-derecha y, si la casilla está ocupada, se corre una a la izquierda.",
}


# -----------------------------------------------------------------------------
# CAPA 2 · LÓGICA
# Funciones "puras": reciben datos y devuelven resultados. No imprimen ni dibujan.
# -----------------------------------------------------------------------------

def cuadro_magico(n: int, lado: str = "arriba") -> np.ndarray:
    """Genera un cuadro mágico n×n con el método siamés.

    Reglas: el 1 va en el centro de la fila de arriba; cada número siguiente va
    una casilla en diagonal arriba a la derecha (dando la vuelta por los bordes);
    si esa casilla ya está ocupada, el número va justo debajo del anterior.
    Para empezar en otro lado, el cuadro se refleja o se gira (sigue siendo mágico).
    """
    if n not in TAMANOS:
        raise ValueError(f"n debe ser impar entre 3 y 9 (llegó {n}).")
    if lado not in LADOS:
        raise ValueError(f"lado debe ser uno de {LADOS} (llegó {lado!r}).")

    M = np.zeros((n, n), dtype=int)
    fila, col = 0, n // 2                      # centro de la fila de arriba
    for numero in range(1, n * n + 1):
        M[fila, col] = numero
        sig_fila, sig_col = (fila - 1) % n, (col + 1) % n   # diagonal; % = dar la vuelta
        if M[sig_fila, sig_col] != 0:                        # ¿ocupada?
            sig_fila, sig_col = (fila + 1) % n, col          # entonces baja una
        fila, col = sig_fila, sig_col

    if lado == "abajo":
        M = np.flipud(M)          # reflejo vertical
    elif lado == "izquierda":
        M = M.T                   # transpuesta
    elif lado == "derecha":
        M = np.rot90(M, -1)       # giro de 90° a la derecha
    return M


def verificar_magico(M: np.ndarray) -> dict:
    """Comprueba que filas, columnas y diagonales sumen n(n² + 1) / 2."""
    n = M.shape[0]
    suma_magica = n * (n * n + 1) // 2
    filas = M.sum(axis=1)
    columnas = M.sum(axis=0)
    diagonales = [int(np.trace(M)), int(np.trace(np.fliplr(M)))]
    usa_todos = sorted(M.flatten().tolist()) == list(range(1, n * n + 1))
    ok = bool(all(filas == suma_magica) and all(columnas == suma_magica)
              and all(d == suma_magica for d in diagonales) and usa_todos)
    return {"suma_magica": suma_magica, "filas": filas.tolist(), "columnas": columnas.tolist(),
            "diagonales": diagonales, "ok": ok}


def pasos_siames(n: int = 5) -> list:
    """Repite el método siamés guardando cada paso y la regla que se usó."""
    M = np.zeros((n, n), dtype=int)
    fila, col = 0, n // 2
    regla, texto = "Inicio", "Se coloca el 1 en el centro de la fila de arriba."
    pasos = []
    for numero in range(1, n * n + 1):
        M[fila, col] = numero
        pasos.append({"numero": numero, "fila": fila, "columna": col, "regla": regla, "texto": texto,
                      "filas": M.sum(axis=1).tolist(), "columnas": M.sum(axis=0).tolist()})
        sale_arriba, sale_derecha = fila - 1 < 0, col + 1 >= n
        sig_fila, sig_col = (fila - 1) % n, (col + 1) % n
        sig = numero + 1
        if M[sig_fila, sig_col] != 0:
            regla = "Ocupada → baja"
            texto = (f"La diagonal (fila {sig_fila + 1}, columna {sig_col + 1}) ya tiene el {M[sig_fila, sig_col]}, "
                     f"así que el {sig} va justo debajo del {numero}.")
            sig_fila, sig_col = (fila + 1) % n, col
        elif sale_arriba:
            regla = "Sale por arriba"
            texto = f"La diagonal se sale por arriba, así que el {sig} aparece abajo, en la columna {sig_col + 1}."
        elif sale_derecha:
            regla = "Sale por la derecha"
            texto = f"La diagonal se sale por la derecha, así que el {sig} aparece a la izquierda, en la fila {sig_fila + 1}."
        else:
            regla = "Diagonal ↗"
            texto = f"El {sig} va una casilla en diagonal arriba a la derecha del {numero}."
        fila, col = sig_fila, sig_col
    return pasos


def matriz_riesgo() -> np.ndarray:
    """Matriz 5×5 de riesgo inherente: cada casilla = frecuencia × impacto."""
    return np.array([[f * i for _, i in IMPACTO] for _, f in FRECUENCIA])


def nivel(valor: float, maximo: float) -> str:
    """Nivel de aceptación según el % del riesgo máximo (umbrales del informe)."""
    porcentaje = valor * 100 / maximo
    for nombre, tope in NIVELES:
        if tope is None or porcentaje <= tope:
            return nombre


def matriz_prioridad(R: np.ndarray):
    """Numera las 25 casillas del 1 (más peligrosa) al 25 (menos peligrosa).

    Desempate: si dos casillas valen lo mismo, va primero la de mayor impacto.
    Devuelve la matriz de prioridades y la lista ordenada de casillas.
    """
    casillas = [(f, c, int(R[f, c])) for f in range(5) for c in range(5)]
    orden = sorted(casillas, key=lambda x: (-x[2], -IMPACTO[x[1]][1], -FRECUENCIA[x[0]][1]))
    P = np.zeros((5, 5), dtype=int)
    for prioridad, (f, c, _) in enumerate(orden, start=1):
        P[f, c] = prioridad
    return P, orden


def pasos_prioridad(R: np.ndarray) -> list:
    """Cada paso coloca una prioridad y explica por qué va ahí."""
    P, orden = matriz_prioridad(R)
    maximo = int(R.max())
    parcial = np.zeros((5, 5), dtype=int)
    pasos = []
    for k, (f, c, v) in enumerate(orden, start=1):
        parcial[f, c] = k
        texto = (f"Prioridad {k}: {FRECUENCIA[f][0]} × {IMPACTO[c][0]} = "
                 f"{FRECUENCIA[f][1]} × {IMPACTO[c][1]} = {v}.")
        siguiente = orden[k] if k < len(orden) else None
        anterior = orden[k - 2] if k > 1 else None
        if siguiente and siguiente[2] == v:
            texto += (f" Empata con otra casilla de {v} ({FRECUENCIA[siguiente[0]][0]} × "
                      f"{IMPACTO[siguiente[1]][0]}); esta va antes porque su impacto es mayor.")
        elif anterior and anterior[2] == v:
            texto += f" Empata con la prioridad {k - 1}, que fue antes por tener mayor impacto."
        pasos.append({"numero": k, "fila": f, "columna": c, "valor": v, "nivel": nivel(v, maximo),
                      "texto": texto, "filas": parcial.sum(axis=1).tolist(),
                      "columnas": parcial.sum(axis=0).tolist()})
    return pasos


def desequilibrio(M: np.ndarray) -> float:
    """Promedio de cuánto se aleja cada suma de fila y columna del valor parejo.

    Es una medida propia de este trabajo: 0 = perfectamente parejo.
    """
    n = M.shape[0]
    parejo = M.sum() / n
    sumas = np.concatenate([M.sum(axis=1), M.sum(axis=0)])
    return round(float(np.mean(np.abs(sumas - parejo))), 1)


def prioridad_rav() -> dict:
    """Valor RAV de los 15 escenarios (riesgo × factor) y su orden de prioridad."""
    valor_f = dict(FRECUENCIA)
    valor_i = dict(IMPACTO)
    maximo = int(matriz_riesgo().max())
    filas = []
    for id_, situacion, fr, im, criterio, categoria in ESCENARIOS:
        r = valor_f[fr] * valor_i[im]
        factor = FACTORES_RAV[categoria][criterio]
        filas.append({"id": id_, "situacion": situacion, "frecuencia": fr, "impacto": im,
                      "criterio": criterio, "categoria": categoria, "riesgo": r,
                      "factor": factor, "rav": round(r * factor, 2), "nivel": nivel(r, maximo)})
    total_r = sum(x["riesgo"] for x in filas)
    total_rav = round(sum(x["rav"] for x in filas), 2)
    for x in filas:
        x["pct"] = round(x["rav"] * 100 / total_rav, 2)
        x["pos_inicial"] = 1 + sum(1 for y in filas if y["riesgo"] > x["riesgo"])  # empates comparten puesto
    filas.sort(key=lambda x: (-x["rav"], -x["riesgo"]))
    for prioridad, x in enumerate(filas, start=1):
        x["prioridad"] = prioridad
    return {"filas": filas, "total_riesgo": total_r, "total_rav": total_rav}


# -----------------------------------------------------------------------------
# CAPA 3 · PRECÁLCULO
# Se calcula TODO de una vez y se empaca en un diccionario (luego JSON).
# Así la interfaz no vuelve a calcular nada y no depende de la sesión de Colab.
# -----------------------------------------------------------------------------

def precalcular() -> dict:
    R = matriz_riesgo()
    maximo = int(R.max())
    P, _ = matriz_prioridad(R)

    magicos = {}
    for n in TAMANOS:
        magicos[str(n)] = {}
        for lado in LADOS:
            M = cuadro_magico(n, lado)
            v = verificar_magico(M)
            magicos[str(n)][lado] = {"matriz": M.tolist(), **v, "regla": REGLAS_LADO[lado],
                                     "desequilibrio": desequilibrio(M)}

    niveles_r = [[nivel(int(R[f, c]), maximo) for c in range(5)] for f in range(5)]
    escala_log = [[round(float(np.log(R[f, c]) / np.log(maximo)), 3) for c in range(5)] for f in range(5)]
    escen_celda = [[[] for _ in range(5)] for _ in range(5)]
    nombres_f = [x[0] for x in FRECUENCIA]
    nombres_i = [x[0] for x in IMPACTO]
    for id_, _, fr, im, _, _ in ESCENARIOS:
        escen_celda[nombres_f.index(fr)][nombres_i.index(im)].append(id_)
    conteo = {nombre: sum(fila.count(nombre) for fila in niveles_r) for nombre, _ in NIVELES}

    return {
        "frecuencia": [{"nombre": a, "valor": b} for a, b in FRECUENCIA],
        "impacto": [{"nombre": a, "valor": b} for a, b in IMPACTO],
        "niveles": [{"nombre": a, "tope": b} for a, b in NIVELES],
        "magicos": magicos,
        "riesgo": {"matriz": R.tolist(), "maximo": maximo, "niveles": niveles_r, "log": escala_log,
                   "escenarios": escen_celda, "conteo": conteo},
        "prioridad": {"matriz": P.tolist(), "filas": P.sum(axis=1).tolist(),
                      "columnas": P.sum(axis=0).tolist(), "desequilibrio": desequilibrio(P)},
        "pasos_magico": pasos_siames(5),
        "pasos_prioridad": pasos_prioridad(R),
        "rav": prioridad_rav(),
        "factores_rav": FACTORES_RAV,
    }


# -----------------------------------------------------------------------------
# CAPA 4 · INTERFAZ
# Una página web (HTML + CSS + JavaScript) que recibe el JSON y lo muestra.
# Aquí NO hay cálculos de riesgo ni del cuadro mágico: solo dibujo y navegación.
# -----------------------------------------------------------------------------

PLANTILLA_HTML = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="color-scheme" content="only light">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Cuadro mágico vs. matriz de riesgo</title>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600&family=IBM+Plex+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
#cmr{color-scheme:only light;--ink:#15202B;--ink2:#33404D;--muted:#4A5663;--quiet:#5B6875;--line:#DDE3EA;--bg:#F4F6F9;--card:#FFFFFF;--accent:#2346A0;--amber:#E0A100;
  font-family:'IBM Plex Sans',system-ui,-apple-system,'Segoe UI',sans-serif;color:var(--ink);background:var(--bg);padding:28px 28px 36px;border-radius:14px;box-sizing:border-box;line-height:1.45;max-width:1320px;margin:0 auto}
#cmr *{box-sizing:border-box}
#cmr .mono{font-family:'IBM Plex Mono',ui-monospace,Menlo,Consolas,monospace}
#cmr h1{margin:0;font-size:28px;font-weight:700;letter-spacing:-0.01em}
#cmr h2{margin:0;font-size:17px;font-weight:600}
#cmr p{margin:0}
#cmr .eyebrow{font-size:12px;font-weight:600;letter-spacing:.08em;text-transform:uppercase;color:var(--quiet)}
#cmr .lead{font-size:15px;color:var(--muted);max-width:800px}
#cmr .head{display:flex;flex-direction:column;gap:10px;margin-bottom:18px}
#cmr .topbar{display:flex;flex-wrap:wrap;align-items:flex-end;justify-content:space-between;gap:10px;border-bottom:1px solid var(--line)}
#cmr .tabs{display:flex;flex-wrap:wrap;gap:2px}
#cmr .tab{appearance:none;border:0;background:none;font:inherit;font-size:14px;font-weight:500;color:var(--muted);min-height:44px;padding:0 14px;cursor:pointer;border-bottom:2px solid transparent;margin-bottom:-1px}
#cmr .tab.on{color:var(--ink);font-weight:600;border-bottom-color:var(--accent)}
#cmr .btn{appearance:none;font:inherit;font-size:14px;font-weight:500;min-height:44px;padding:0 16px;border-radius:8px;border:1px solid #C9D2DC;background:#fff;color:var(--ink);cursor:pointer}
#cmr .btn.primary{background:var(--accent);border-color:var(--accent);color:#fff;font-weight:600}
#cmr .btn:disabled{opacity:.45;cursor:default}
#cmr button:focus-visible,#cmr select:focus-visible,#cmr input:focus-visible{outline:2px solid var(--accent);outline-offset:2px}
#cmr .stack{display:flex;flex-direction:column;gap:20px}
#cmr .card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:20px;min-width:0}
#cmr .cards3{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:18px;align-items:start}
#cmr .cards2{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:18px;align-items:start}
#cmr .kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:14px}
#cmr .kpi{background:#fff;border:1px solid var(--line);border-radius:12px;padding:14px 16px;display:flex;flex-direction:column;gap:4px}
#cmr .kpi .t{font-size:12px;color:var(--quiet)}
#cmr .kpi .v{font-size:26px;font-weight:600}
#cmr .kpi .v small{font-size:15px;color:var(--quiet);font-weight:500}
#cmr .kpi .f{font-size:12px;color:var(--muted)}
#cmr .ok{color:#1E6B3A}
#cmr .cardhead{display:flex;justify-content:space-between;align-items:flex-start;gap:8px;flex-wrap:wrap;margin-bottom:14px}
#cmr .chip{font-size:12px;font-weight:500;padding:4px 10px;border-radius:999px;white-space:nowrap}
#cmr .note{font-size:12.5px;color:var(--muted);margin-top:14px}
#cmr .controls{display:flex;flex-wrap:wrap;align-items:flex-end;gap:16px 26px}
#cmr .ctl{display:flex;flex-direction:column;gap:6px}
#cmr .ctl>span,#cmr .ctl>label{font-size:13px;font-weight:500}
#cmr .seg{display:flex;gap:6px;flex-wrap:wrap}
#cmr .pill{appearance:none;font:inherit;font-size:14px;font-weight:500;min-height:44px;min-width:56px;padding:0 12px;border-radius:8px;border:1px solid #C9D2DC;background:#fff;color:var(--ink);cursor:pointer}
#cmr .pill.on{background:var(--accent);border-color:var(--accent);color:#fff;font-weight:600}
#cmr select{font:inherit;font-size:14px;height:44px;border-radius:8px;border:1px solid #C9D2DC;padding:0 10px;background:#fff;color:var(--ink)}
#cmr .check{display:flex;align-items:center;gap:10px;min-height:44px;font-size:14px;cursor:pointer}
#cmr .check input{width:18px;height:18px;accent-color:var(--accent);margin:0}
#cmr .g{display:grid;gap:4px}
#cmr .c{display:flex;flex-direction:column;align-items:center;justify-content:center;gap:2px;aspect-ratio:1/1;min-width:0;border-radius:6px;font-family:'IBM Plex Mono',ui-monospace,Menlo,Consolas,monospace;line-height:1}
#cmr .c small{font-size:9.5px;opacity:.92;text-align:center;line-height:1.1}
#cmr .c.vacia{background:#F7F9FB;border:1px dashed #C9D2DC;color:var(--quiet)}
#cmr .c.suma{background:#F7F9FB;border:1px dashed #B8C3CF;font-weight:600}
#cmr .lb{display:flex;flex-direction:column;justify-content:center;gap:2px;min-width:0;font-size:11px;font-weight:600;color:var(--ink2);line-height:1.2}
#cmr .lb small{font-family:'IBM Plex Mono',ui-monospace,monospace;font-weight:400;font-size:10px;color:var(--quiet)}
#cmr .lb.mid{align-items:center;text-align:center}
#cmr .explica{border-radius:10px;padding:14px 16px;display:flex;flex-direction:column;gap:4px;min-height:70px;margin-top:16px}
#cmr .explica b{font-size:12px}
#cmr .explica span{font-size:14px;color:var(--ink2)}
#cmr .bars{position:relative;height:160px;display:flex;align-items:flex-end;gap:8px;border-bottom:1px solid #C9D2DC}
#cmr .bar{flex:1 1 0;min-width:0;display:flex;flex-direction:column;align-items:center;justify-content:flex-end;gap:4px;height:100%}
#cmr .bar span{font-size:11px;color:var(--ink2)}
#cmr .bar i{display:block;width:100%;max-width:40px;border-radius:4px 4px 0 0}
#cmr .barnames{display:flex;gap:8px;margin-top:6px}
#cmr .barnames span{flex:1 1 0;min-width:0;text-align:center;font-size:11px;color:var(--quiet)}
#cmr .ref{position:absolute;left:0;right:0;border-top:2px dashed var(--ink)}
#cmr .tiles{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:14px}
#cmr .tile{background:#F7F9FB;border-radius:10px;padding:14px 16px;display:flex;flex-direction:column;gap:6px;font-size:13px;color:var(--muted)}
#cmr .tile b{color:var(--ink);font-size:13px}
#cmr .aviso{font-size:13px;padding:12px 14px;border-radius:8px;background:#FFF6DB;color:#5C4500}
#cmr .legend{display:flex;flex-wrap:wrap;gap:12px 24px;align-items:center;font-size:12px;color:var(--ink2)}
#cmr .legend span{display:inline-flex;align-items:center;gap:7px}
#cmr .sw{display:inline-block;width:12px;height:12px;border-radius:3px;border:1px solid rgba(21,32,43,.18)}
#cmr .grad{display:inline-block;width:90px;height:12px;border-radius:4px;border:1px solid rgba(21,32,43,.12)}
#cmr .stepbar{display:flex;flex-wrap:wrap;align-items:center;gap:12px 18px}
#cmr .stepbar .btns{display:flex;flex-wrap:wrap;gap:8px}
#cmr .stepbar .rng{flex:1 1 240px;display:flex;align-items:center;gap:14px;min-width:0}
#cmr .stepbar input[type=range]{flex:1 1 auto;min-width:0;accent-color:var(--accent);height:44px}
#cmr .done{background:var(--ink);color:#fff;border-radius:12px;padding:16px 20px;display:flex;flex-direction:column;gap:4px}
#cmr .done span{color:#DCE3EA;font-size:14px}
#cmr table{border-collapse:collapse;width:100%;font-size:13px}
#cmr th{text-align:left;font-size:12px;font-weight:600;color:var(--quiet);padding:8px 10px;border-bottom:1px solid var(--line);white-space:nowrap}
#cmr td{padding:9px 10px;border-bottom:1px solid #EEF1F4;vertical-align:middle}
#cmr td.num,#cmr th.num{text-align:right;font-family:'IBM Plex Mono',ui-monospace,monospace;white-space:nowrap}
#cmr .tbl{overflow-x:auto}
#cmr .tag{display:inline-block;font-size:11.5px;font-weight:600;padding:3px 8px;border-radius:6px}
#cmr .pbar{display:flex;align-items:center;gap:8px;min-width:140px}
#cmr .pbar i{display:block;height:10px;border-radius:3px;background:#D9573A;flex:none}
#cmr .pbar span{white-space:nowrap}
#cmr .slide{display:flex;flex-direction:column;gap:18px}
#cmr .dots{display:flex;gap:6px}
#cmr .dots i{width:9px;height:9px;border-radius:50%;background:#C9D2DC}
#cmr .dots i.on{background:var(--accent)}
#cmr .rules{margin:0;padding-left:20px;font-size:14.5px;color:var(--ink2);display:flex;flex-direction:column;gap:8px}
#cmr .factores td,#cmr .factores th{text-align:center}
#cmr .factores td:first-child,#cmr .factores th:first-child{text-align:left}
</style>
</head>
<body style="margin:0;background:#F4F6F9">
<div id="cmr" aria-live="polite"></div>
<script>
(function () {
  "use strict";
  // Datos calculados por Python. Esta página NO calcula: solo muestra.
  const D = __DATOS__;

  const root = document.getElementById("cmr");
  const S = { vista: "general", n: 5, lado: "arriba", esc: true, color: "niveles", k: 0, slide: 0, play: false };
  let timer = null;

  const COL_NIVEL = {
    Bajo: { bg: "#E8F3E5", fg: "#1E4620" },
    Mediano: { bg: "#BFDDF0", fg: "#123F5C" },
    Alto: { bg: "#F0B429", fg: "#3D2C00" },
    Extremo: { bg: "#C8372D", fg: "#FFFFFF" }
  };
  const TOPE_TXT = { Bajo: "≤ 2 %", Mediano: "≤ 5 %", Alto: "≤ 20 %", Extremo: "> 20 %" };
  const MAG = ["#F2F5FD", "#C9D5F3", "#8FA8E6"];
  const CONT = ["#FFF6E8", "#F8C98A", "#F09A60", "#E0664A"];
  const ABR = ["Inf.", "Men.", "Imp.", "May.", "Sup."];
  const FS = { 3: 24, 5: 18, 7: 14, 9: 11 };

  // ---------- utilidades de dibujo ----------
  const hex = h => [1, 3, 5].map(i => parseInt(h.slice(i, i + 2), 16));
  function ramp(stops, t) {
    t = Math.max(0, Math.min(1, t));
    const seg = (stops.length - 1) * t, i = Math.min(Math.floor(seg), stops.length - 2), f = seg - i;
    const a = hex(stops[i]), b = hex(stops[i + 1]);
    return "rgb(" + a.map((v, j) => Math.round(v + (b[j] - v) * f)).join(",") + ")";
  }
  const num = x => String(x).replace(".", ",");
  const esc = s => String(s).replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/"/g, "&quot;");

  function celda(main, sub, bg, fg, fs, extra, title) {
    return '<div class="c" style="background:' + bg + ";color:" + fg + ";" + (extra || "") + '" title="' + esc(title || "") + '">' +
      '<span style="font-size:' + fs + 'px;font-weight:600">' + main + "</span>" + (sub ? "<small>" + sub + "</small>" : "") + "</div>";
  }
  const vacia = (txt, fs) => '<div class="c vacia"><span style="font-size:' + (fs || 13) + 'px">' + (txt || "") + "</span></div>";
  const suma = (v, fs, ok) => '<div class="c suma" style="color:' + (ok ? "#1E6B3A" : "#15202B") + '" title="Suma"><span style="font-size:' + fs + 'px">' + v + "</span></div>";
  const etiqueta = (main, sub, mid, title) => '<div class="lb' + (mid ? " mid" : "") + '" title="' + esc(title || main) + '">' + main + (sub ? "<small>" + sub + "</small>" : "") + "</div>";
  const anillo = color => "box-shadow:0 0 0 2.5px " + color + ";";

  // ---------- cuadrículas ----------
  // Cuadro mágico. hasta = null muestra todo; hasta = k muestra solo 1..k (paso a paso).
  function gridMagico(info, hasta, sumasFilas, sumasCols) {
    const M = info.matriz, n = M.length, N2 = n * n, fs = FS[n] || 14;
    const completo = hasta === null || hasta === N2;
    const filas = sumasFilas || info.filas, cols = sumasCols || info.columnas;
    let h = '<div class="g" style="grid-template-columns:repeat(' + (n + 1) + ',minmax(0,1fr));gap:' + (n >= 7 ? 3 : 4) + 'px">';
    for (let i = 0; i < n; i++) {
      for (let j = 0; j < n; j++) {
        const v = M[i][j];
        if (hasta !== null && v > hasta) { h += vacia(); continue; }
        const marca = hasta === null ? (v === 1 || v === N2) : v === hasta;
        h += celda(v, "", ramp(MAG, (v - 1) / (N2 - 1)), "#15202B", fs, marca ? anillo("var(--amber)") : "", "Fila " + (i + 1) + ", columna " + (j + 1) + ": " + v);
      }
      h += suma(filas[i], Math.max(11, fs - 3), completo && filas[i] === info.suma_magica);
    }
    cols.forEach(v => { h += suma(v, Math.max(11, fs - 3), completo && v === info.suma_magica); });
    h += etiqueta("Σ", "", true, "Sumas");
    return h + "</div>";
  }

  function cabeceraMatriz(conSuma) {
    let h = etiqueta("F \\ I", "", true, "Frecuencia (filas) por impacto (columnas)");
    D.impacto.forEach((x, c) => { h += etiqueta(ABR[c], "I=" + x.valor, true, x.nombre); });
    if (conSuma) h += etiqueta("Σ", "", true, "Suma de la fila");
    return h;
  }

  function gridRiesgo() {
    const R = D.riesgo;
    let h = '<div class="g" style="grid-template-columns:86px repeat(5,minmax(0,1fr))">' + cabeceraMatriz(false);
    for (let f = 0; f < 5; f++) {
      h += etiqueta(D.frecuencia[f].nombre, "F=" + D.frecuencia[f].valor, false);
      for (let c = 0; c < 5; c++) {
        const v = R.matriz[f][c], niv = R.niveles[f][c];
        let bg = COL_NIVEL[niv].bg, fg = COL_NIVEL[niv].fg;
        if (S.color === "continuo") { bg = ramp(CONT, R.log[f][c]); fg = "#15202B"; }
        const ids = S.esc ? R.escenarios[f][c].join(" ") : "";
        h += celda(v, ids, bg, fg, 15, "", D.frecuencia[f].nombre + " × " + D.impacto[c].nombre + " = " + v + " · " + niv + (ids ? " · " + ids : ""));
      }
    }
    return h + "</div>";
  }

  // Cuadro de prioridades. hasta = null muestra todo; hasta = k muestra 1..k.
  function gridPrioridad(hasta, sumasFilas, sumasCols) {
    const P = D.prioridad.matriz, R = D.riesgo;
    const filas = sumasFilas || D.prioridad.filas, cols = sumasCols || D.prioridad.columnas;
    let h = '<div class="g" style="grid-template-columns:86px repeat(6,minmax(0,1fr))">' + cabeceraMatriz(true);
    for (let f = 0; f < 5; f++) {
      h += etiqueta(D.frecuencia[f].nombre, "F=" + D.frecuencia[f].valor, false);
      for (let c = 0; c < 5; c++) {
        const p = P[f][c], v = R.matriz[f][c], col = COL_NIVEL[R.niveles[f][c]];
        if (hasta !== null && p > hasta) { h += vacia(v, 12); continue; }
        const marca = hasta === null ? p <= 5 : p === hasta;
        h += celda(p, "R " + v, col.bg, col.fg, 17, marca ? anillo(hasta === null ? "#15202B" : "var(--amber)") : "", "Prioridad " + p + ": " + D.frecuencia[f].nombre + " × " + D.impacto[c].nombre + " = " + v);
      }
      h += suma(filas[f], 14, false);
    }
    h += etiqueta("Σ", "", false, "Suma de la columna");
    cols.forEach(v => { h += suma(v, 14, false); });
    return h + "<div></div></div>";
  }

  function barras(sumas, nombres, color, parejo) {
    const mx = Math.max.apply(null, sumas);
    let h = '<div class="bars"><div class="ref" style="bottom:' + Math.round(parejo / mx * 130) + 'px" title="Valor parejo: ' + parejo + '"></div>';
    sumas.forEach(v => { h += '<div class="bar"><span class="mono">' + v + '</span><i style="height:' + Math.max(4, Math.round(v / mx * 130)) + "px;background:" + color + '"></i></div>'; });
    h += '</div><div class="barnames">' + nombres.map(x => "<span>" + x + "</span>").join("") + "</div>";
    return h;
  }

  function leyenda() {
    let h = '<div class="legend"><span><i class="grad" style="background:linear-gradient(90deg,' + MAG[0] + "," + MAG[2] + ')"></i>Cuadro mágico: 1 → ' + (S.n * S.n) + "</span>";
    D.niveles.forEach(l => { h += '<span><i class="sw" style="background:' + COL_NIVEL[l.nombre].bg + '"></i>' + l.nombre + " (" + TOPE_TXT[l.nombre] + ")</span>"; });
    if (S.color === "continuo") h += '<span><i class="grad" style="background:linear-gradient(90deg,' + CONT.join(",") + ')"></i>Matriz en color continuo (escala log.)</span>';
    return h + "</div>";
  }

  // ---------- encabezado y pestañas ----------
  const VISTAS = [["general", "Vista general"], ["pasos", "Paso a paso"], ["rav", "Escenarios RAV"]];
  function encabezado() {
    let tabs = VISTAS.map(v => '<button type="button" class="tab' + (S.vista === v[0] ? " on" : "") + '" data-act="vista" data-v="' + v[0] + '"' + (S.vista === v[0] ? ' aria-current="page"' : "") + ">" + v[1] + "</button>").join("");
    return '<div class="head"><span class="eyebrow">Protocolos de Comunicación y Seguridad Informática · TEINCO</span>' +
      "<h1>Cuadro mágico vs. matriz de riesgo</h1>" +
      '<p class="lead">Dos formas de acomodar los números en una cuadrícula: una los reparte para que todo sume igual; la otra los ordena por peligro para saber qué atender primero.</p>' +
      '<div class="topbar"><nav class="tabs" aria-label="Vistas">' + tabs + '</nav><button type="button" class="btn primary" data-act="presentar" style="margin-bottom:6px">Modo sustentación</button></div></div>';
  }

  // ---------- vista general ----------
  function vistaGeneral() {
    const info = D.magicos[String(S.n)][S.lado];
    const pills = [3, 5, 7, 9].map(n => '<button type="button" class="pill' + (S.n === n ? " on" : "") + '" aria-pressed="' + (S.n === n) + '" data-act="n" data-v="' + n + '">' + n + "×" + n + "</button>").join("");
    const colores = [["niveles", "Niveles"], ["continuo", "Continuo"]].map(m => '<button type="button" class="pill' + (S.color === m[0] ? " on" : "") + '" aria-pressed="' + (S.color === m[0]) + '" data-act="color" data-v="' + m[0] + '">' + m[1] + "</button>").join("");
    const lados = [["arriba", "Centro de arriba (clásico)"], ["abajo", "Centro de abajo"], ["izquierda", "Centro de la izquierda"], ["derecha", "Centro de la derecha"]]
      .map(o => '<option value="' + o[0] + '"' + (S.lado === o[0] ? " selected" : "") + ">" + o[1] + "</option>").join("");

    let h = '<div class="stack">';
    h += '<section class="card"><div class="controls">' +
      '<div class="ctl"><span>Tamaño del cuadro mágico</span><div class="seg">' + pills + "</div></div>" +
      '<div class="ctl"><label for="cmr-lado">Lado donde empieza el 1</label><select id="cmr-lado" data-act="lado">' + lados + "</select></div>" +
      '<div class="ctl"><span>Color de la matriz</span><div class="seg">' + colores + "</div></div>" +
      '<label class="check"><input type="checkbox" data-act="esc"' + (S.esc ? " checked" : "") + "> Mostrar escenarios R01–R15</label>" +
      "</div></section>";

    h += '<section class="kpis" aria-label="Indicadores">' +
      '<div class="kpi"><span class="t">Suma mágica (n = ' + S.n + ')</span><span class="v mono">' + info.suma_magica + '</span><span class="f ' + (info.ok ? "ok" : "") + '">' + (info.ok ? "Verificado: filas, columnas y diagonales" : "Error: las sumas no coinciden") + "</span></div>" +
      '<div class="kpi"><span class="t">Riesgo máximo de la matriz</span><span class="v mono">' + D.riesgo.maximo + '</span><span class="f">Muy frecuente × Superior</span></div>' +
      '<div class="kpi"><span class="t">Casillas en nivel extremo</span><span class="v mono">' + D.riesgo.conteo.Extremo + '<small> / 25</small></span><span class="f">Más del 20 % del riesgo máximo</span></div>' +
      '<div class="kpi"><span class="t">Desequilibrio de sumas</span><span class="v mono">' + num(info.desequilibrio) + "<small> vs </small>" + num(D.prioridad.desequilibrio) + '</span><span class="f">Mágico vs. prioridades (desviación media)</span></div>' +
      "</section>";

    h += '<section class="cards3">' +
      '<article class="card"><div class="cardhead"><div><span class="eyebrow">Modelo A</span><h2>Cuadro mágico ' + S.n + "×" + S.n + '</h2></div><span class="chip" style="background:#EEF2FB;color:#1B2F73">Reparte parejo</span></div>' +
      gridMagico(info, null) + '<p class="note">' + info.regla + " El anillo amarillo marca el 1 y el " + (S.n * S.n) + ": siempre quedan en lados opuestos.</p></article>" +
      '<article class="card"><div class="cardhead"><div><span class="eyebrow">Modelo B</span><h2>Matriz de riesgo</h2></div><span class="chip" style="background:#FBEDEA;color:#7A2018">R = F × I</span></div>' +
      gridRiesgo() + '<p class="note">Cada casilla es frecuencia × impacto. ' + (S.color === "continuo" ? "Más oscuro = más riesgo (escala logarítmica)." : "El color es el nivel de aceptación.") + (S.esc ? " Los códigos son los escenarios del informe." : "") + "</p></article>" +
      '<article class="card"><div class="cardhead"><div><span class="eyebrow">Modelo C</span><h2>Cuadro de prioridades</h2></div><span class="chip" style="background:#FFF3D6;color:#5C4500">Ordena por peligro</span></div>' +
      gridPrioridad(null) + '<p class="note">Número grande = prioridad (1 = mayor riesgo); número pequeño = valor R. Las 5 primeras llevan contorno oscuro. En un empate gana la de mayor impacto.</p></article>' +
      "</section>";

    h += '<section class="card stack"><div><h2>Comparación: ¿cómo se reparten los números?</h2><p class="lead" style="font-size:14px;margin-top:6px">Suma de cada columna. En el cuadro mágico todas miden lo mismo. En el de prioridades los números pequeños (lo urgente) se acumulan en las columnas de mayor impacto, así que esas columnas suman menos. La línea punteada es el valor parejo.</p></div>' +
      '<div class="cards2" style="gap:28px"><figure style="margin:0"><figcaption style="font-size:13px;font-weight:600;margin-bottom:8px">A · Cuadro mágico ' + S.n + "×" + S.n + ' <span style="font-weight:400;color:var(--quiet)">(parejo = ' + info.suma_magica + ")</span></figcaption>" +
      barras(info.columnas, info.columnas.map((_, j) => "Col " + (j + 1)), "#5C7BD6", info.suma_magica) + "</figure>" +
      '<figure style="margin:0"><figcaption style="font-size:13px;font-weight:600;margin-bottom:8px">C · Cuadro de prioridades 5×5 <span style="font-weight:400;color:var(--quiet)">(parejo = 65)</span></figcaption>' +
      barras(D.prioridad.columnas, ABR, "#D9573A", 65) + "</figure></div>" +
      (S.n !== 5 ? '<div class="aviso">Con n = ' + S.n + " el cuadro mágico usa los números del 1 al " + (S.n * S.n) + " y la matriz de riesgo sigue en 5×5 (del 1 al 25). Se ven las dos formas de repartir, pero la comparación número a número solo aplica con n = 5.</div>" : "") +
      '<div class="tiles"><div class="tile"><b>A · Cuadro mágico</b>Regla de recorrido: diagonal y, si está ocupada, bajar. Filas, columnas y diagonales suman lo mismo.</div>' +
      '<div class="tile"><b>B · Matriz de riesgo</b>Relaciona qué tan seguido pasa algo con qué tan grave es. El impacto crece más rápido, por eso pesa más.</div>' +
      '<div class="tile"><b>C · Cuadro de prioridades</b>Usa el mismo 1–25 que el mágico, pero acomodado por peligro. No suma parejo, y eso es lo correcto.</div></div>' +
      "</section>";

    h += leyenda() + "</div>";
    return h;
  }

  // ---------- paso a paso ----------
  function barraPasos() {
    return '<section class="card stepbar" style="padding:14px 18px"><div class="btns">' +
      '<button type="button" class="btn" data-act="k0"' + (S.k === 0 ? " disabled" : "") + ">Reiniciar</button>" +
      '<button type="button" class="btn" data-act="kmenos"' + (S.k === 0 ? " disabled" : "") + ">Anterior</button>" +
      '<button type="button" class="btn primary" data-act="play" style="min-width:120px">' + (S.play ? "Pausar" : "Reproducir") + "</button>" +
      '<button type="button" class="btn" data-act="kmas"' + (S.k === 25 ? " disabled" : "") + ">Siguiente</button>" +
      '<button type="button" class="btn" data-act="kfin"' + (S.k === 25 ? " disabled" : "") + ">Final</button></div>" +
      '<div class="rng"><input type="range" min="0" max="25" step="1" value="' + S.k + '" data-act="rango" aria-label="Paso actual"><span class="mono" id="cmr-paso" style="font-size:15px;font-weight:600;white-space:nowrap">Paso ' + S.k + " de 25</span></div></section>";
  }

  function tarjetaPasoMagico() {
    const info = D.magicos["5"].arriba, p = S.k > 0 ? D.pasos_magico[S.k - 1] : null;
    const cero = [0, 0, 0, 0, 0];
    return '<article class="card"><div class="cardhead"><div><span class="eyebrow">Modelo A · método siamés</span><h2>Cuadro mágico 5×5</h2></div><span class="chip" style="background:#EEF2FB;color:#1B2F73">' + (p ? p.regla : "Sin empezar") + "</span></div>" +
      '<div style="max-width:420px;margin:0 auto">' + gridMagico(info, S.k, p ? p.filas : cero, p ? p.columnas : cero) + "</div>" +
      '<div class="explica" style="background:#F4F7FD"><b style="color:#1B2F73">Qué pasó en este paso</b><span>' +
      (p ? "Número " + p.numero + " en la fila " + (p.fila + 1) + ", columna " + (p.columna + 1) + ". " + p.texto : "Pulsa Siguiente o Reproducir. El 1 irá en el centro de la fila de arriba.") + "</span></div></article>";
  }

  function tarjetaPasoPrioridad() {
    const p = S.k > 0 ? D.pasos_prioridad[S.k - 1] : null;
    const cero = [0, 0, 0, 0, 0];
    return '<article class="card"><div class="cardhead"><div><span class="eyebrow">Modelo C · el más peligroso primero</span><h2>Cuadro de prioridades 5×5</h2></div><span class="chip" style="background:#FFF3D6;color:#5C4500">' + (p ? "Nivel " + p.nivel : "Sin empezar") + "</span></div>" +
      '<div style="max-width:470px;margin:0 auto">' + gridPrioridad(S.k, p ? p.filas : cero, p ? p.columnas : cero) + "</div>" +
      '<div class="explica" style="background:#FFF8E8"><b style="color:#5C4500">Qué pasó en este paso</b><span>' +
      (p ? p.texto : "Pulsa Siguiente o Reproducir. Primero se elige la casilla de mayor riesgo; las grises muestran el valor R que falta ordenar.") + "</span></div></article>";
  }

  function zonaPasos() {
    let h = '<section class="cards2">' + tarjetaPasoMagico() + tarjetaPasoPrioridad() + "</section>";
    if (S.k === 25) {
      const cols = D.prioridad.columnas;
      h += '<section class="done"><b style="font-size:16px">Terminado: los mismos 25 números, dos resultados distintos</b><span>En el cuadro mágico el 25 quedó en el centro de abajo, el lado opuesto al 1, y todas las sumas dan 65. En el cuadro de prioridades las columnas suman de ' + Math.min.apply(null, cols) + " a " + Math.max.apply(null, cols) + ": lo urgente se concentra donde el impacto es mayor.</span></section>";
    }
    return h;
  }

  function vistaPasos() {
    return '<div class="stack">' + barraPasos() + '<div id="cmr-zona" class="stack">' + zonaPasos() + "</div>" +
      '<p class="note" style="margin:0">Anillo amarillo: el número que se acaba de colocar. Fila y columna punteadas: sumas parciales hasta este paso.</p></div>';
  }

  // ---------- escenarios RAV ----------
  function tablaRav(limite) {
    const filas = D.rav.filas.slice(0, limite || 15), mx = D.rav.filas[0].pct;
    let h = '<div class="tbl"><table><thead><tr><th class="num">Prioridad</th><th>ID</th><th>Escenario</th><th class="num">R</th><th>Catalogación RAV</th><th class="num">Factor</th><th class="num">Valor RAV</th><th>% del total</th><th>Nivel</th><th class="num">Antes</th></tr></thead><tbody>';
    filas.forEach(x => {
      const col = COL_NIVEL[x.nivel];
      h += "<tr><td class=\"num\" style=\"font-weight:600\">" + x.prioridad + '</td><td class="mono">' + x.id + "</td><td>" + esc(x.situacion) + '</td><td class="num">' + x.riesgo + "</td><td>" + x.categoria + " (" + x.criterio.toLowerCase() + ')</td><td class="num">× ' + num(x.factor) + '</td><td class="num" style="font-weight:600">' + num(x.rav) + "</td>" +
        '<td><div class="pbar"><i style="width:' + Math.max(2, Math.round(x.pct / mx * 110)) + 'px"></i><span class="mono">' + num(x.pct) + " %</span></div></td>" +
        '<td><span class="tag" style="background:' + col.bg + ";color:" + col.fg + '">' + x.nivel + '</span></td><td class="num">' + x.pos_inicial + "</td></tr>";
    });
    return h + "</tbody></table></div>";
  }

  function vistaRav() {
    const cats = Object.keys(D.factores_rav), crits = ["Percibido", "Identificado", "Verificado", "No aplicado"];
    let fac = '<div class="tbl"><table class="factores"><thead><tr><th>Categoría</th>' + crits.map(c => "<th>" + c + "</th>").join("") + "</tr></thead><tbody>";
    cats.forEach(cat => { fac += "<tr><td>" + cat + "</td>" + crits.map(c => '<td class="mono">' + num(D.factores_rav[cat][c]) + "</td>").join("") + "</tr>"; });
    fac += "</tbody></table></div>";
    return '<div class="stack"><section class="card"><h2>¿Qué es RAV?</h2><p class="lead" style="font-size:14.5px;margin-top:8px">La matriz solo mira frecuencia e impacto. RAV agrega una pregunta más: <b>¿qué tan comprobado está ese riesgo?</b> Cada escenario se multiplica por un factor. Si está verificado con evidencia pesa más (hasta × 3,2); si alguien solo lo percibe se queda igual (× 1); si ya tiene controles (filtrado) casi desaparece. Por eso el orden cambia: R12, con 9 cuentas de exempleados comprobadas, sube del 4.º al 2.º lugar.</p></section>' +
      '<section class="kpis"><div class="kpi"><span class="t">Riesgo inherente total</span><span class="v mono">' + D.rav.total_riesgo + '</span><span class="f">Suma de F × I de los 15 escenarios</span></div>' +
      '<div class="kpi"><span class="t">Valor RAV total</span><span class="v mono">' + num(D.rav.total_rav) + '</span><span class="f">Después de aplicar los factores</span></div>' +
      '<div class="kpi"><span class="t">Primera prioridad</span><span class="v mono">' + D.rav.filas[0].id + "<small> · " + num(D.rav.filas[0].pct) + ' %</small></span><span class="f">' + esc(D.rav.filas[0].situacion) + "</span></div></section>" +
      '<section class="card"><div class="cardhead"><h2>Prioridad de los 15 escenarios</h2><span class="chip" style="background:#EEF1F4;color:#4A5663">Equivale a la Tabla 5 del informe</span></div>' + tablaRav() +
      '<p class="note">"Antes" es el puesto que tenía cada escenario solo con F × I; los empates comparten puesto.</p></section>' +
      '<section class="card"><h2 style="margin-bottom:12px">Factores RAV (Tabla 3 del informe)</h2>' + fac + "</section></div>";
  }

  // ---------- modo sustentación ----------
  const SLIDES = [
    { t: "1 · El reto",
      c: () => { const i3 = D.magicos["3"].arriba, i5 = D.magicos["5"].arriba; return '<div class="cards2"><article class="card"><h2 style="margin-bottom:12px">Las 4 reglas</h2><ol class="rules"><li>El 1 va en el centro de la fila de arriba.</li><li>Cada número siguiente va en diagonal arriba a la derecha.</li><li>Si se sale por un borde, aparece por el borde contrario.</li><li>Si la casilla está ocupada, va justo debajo del anterior.</li></ol><p class="note">Suma mágica = n(n² + 1) / 2 → 15 en 3×3, 65 en 5×5.</p></article><article class="card"><div class="cards2" style="grid-template-columns:repeat(auto-fit,minmax(160px,1fr))"><div><h2 style="margin-bottom:10px">3×3 · suma 15</h2>' + gridMagico(i3, null) + '</div><div><h2 style="margin-bottom:10px">5×5 · suma 65</h2>' + gridMagico(i5, null) + "</div></div></article></div>"; } },
    { t: "2 · El cuadro mágico, paso a paso", pasos: "magico" },
    { t: "3 · La matriz de riesgo del informe",
      c: () => '<div class="cards2"><article class="card">' + gridRiesgo() + "</article>" + '<article class="card stack" style="gap:12px"><h2>Cómo leerla</h2><p class="lead" style="font-size:14px">Filas = frecuencia (1 a 5). Columnas = impacto (1 a 50). El máximo es 5 × 50 = 250.</p><p class="lead" style="font-size:14px">El color sale del porcentaje respecto a 250: bajo hasta 2 %, mediano hasta 5 %, alto hasta 20 % y extremo por encima.</p>' + leyenda() + "</article></div>" },
    { t: "4 · El cuadro de prioridades, paso a paso", pasos: "prioridad" },
    { t: "5 · La comparación",
      c: () => { const i5 = D.magicos["5"].arriba; return '<div class="cards2"><article class="card"><h2 style="margin-bottom:12px">Cuadro mágico: todo suma 65</h2>' + gridMagico(i5, null) + '<div style="margin-top:18px">' + barras(i5.columnas, i5.columnas.map((_, j) => "Col " + (j + 1)), "#5C7BD6", 65) + '</div></article><article class="card"><h2 style="margin-bottom:12px">Prioridades: lo urgente se concentra</h2>' + gridPrioridad(null) + '<div style="margin-top:18px">' + barras(D.prioridad.columnas, ABR, "#D9573A", 65) + "</div></article></div>" +
        '<section class="kpis"><div class="kpi"><span class="t">Desequilibrio del cuadro mágico</span><span class="v mono">' + num(i5.desequilibrio) + '</span></div><div class="kpi"><span class="t">Desequilibrio del cuadro de prioridades</span><span class="v mono">' + num(D.prioridad.desequilibrio) + "</span></div></section>"; } },
    { t: "6 · Cierre: conexión con el informe",
      c: () => '<section class="card"><div class="cardhead"><h2>Las 5 primeras prioridades (RAV)</h2><span class="chip" style="background:#EEF1F4;color:#4A5663">Tabla completa en la pestaña Escenarios RAV</span></div>' + tablaRav(5) + "</section>" }
  ];

  function vistaPresentacion() {
    const s = SLIDES[S.slide];
    const dots = SLIDES.map((_, i) => '<i class="' + (i === S.slide ? "on" : "") + '"></i>').join("");
    let cuerpo;
    if (s.pasos) {
      cuerpo = barraPasos() + '<div id="cmr-zona" class="stack">' + (s.pasos === "magico" ? '<div style="max-width:620px;width:100%;margin:0 auto">' + tarjetaPasoMagico() + "</div>" : '<div style="max-width:680px;width:100%;margin:0 auto">' + tarjetaPasoPrioridad() + "</div>") + "</div>";
    } else {
      cuerpo = s.c();
    }
    return '<div class="slide"><div style="display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap"><h1 style="font-size:24px">' + s.t + '</h1><div style="display:flex;align-items:center;gap:14px"><div class="dots" aria-hidden="true">' + dots + '</div><span class="mono" style="font-size:13px;color:var(--quiet)">' + (S.slide + 1) + " / " + SLIDES.length + "</span></div></div>" +
      cuerpo +
      '<div style="display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap;border-top:1px solid var(--line);padding-top:16px">' +
      '<button type="button" class="btn" data-act="salir">Salir de la presentación</button><div style="display:flex;gap:8px">' +
      '<button type="button" class="btn" data-act="sant"' + (S.slide === 0 ? " disabled" : "") + ">Anterior</button>" +
      '<button type="button" class="btn primary" data-act="ssig"' + (S.slide === SLIDES.length - 1 ? " disabled" : "") + ">Siguiente</button></div></div></div>";
  }

  // ---------- render y eventos ----------
  function ajustarAltura() {
    try { if (window.google && google.colab && google.colab.output) google.colab.output.setIframeHeight(0, true, { maxHeight: 6000 }); } catch (e) { /* fuera de Colab no hace falta */ }
  }

  function render() {
    let h;
    if (S.vista === "presentacion") h = vistaPresentacion();
    else {
      h = encabezado();
      if (S.vista === "general") h += vistaGeneral();
      else if (S.vista === "pasos") h += vistaPasos();
      else h += vistaRav();
    }
    root.innerHTML = h;
    ajustarAltura();
  }

  // Actualiza solo los cuadros del paso a paso (para no reiniciar el deslizador).
  function renderZona() {
    const zona = document.getElementById("cmr-zona");
    if (!zona) return render();
    if (S.vista === "presentacion") {
      const s = SLIDES[S.slide];
      zona.innerHTML = s.pasos === "magico" ? '<div style="max-width:620px;width:100%;margin:0 auto">' + tarjetaPasoMagico() + "</div>" : '<div style="max-width:680px;width:100%;margin:0 auto">' + tarjetaPasoPrioridad() + "</div>";
    } else zona.innerHTML = zonaPasos();
    const lbl = document.getElementById("cmr-paso");
    if (lbl) lbl.textContent = "Paso " + S.k + " de 25";
    const rango = root.querySelector('[data-act="rango"]');
    if (rango && document.activeElement !== rango) rango.value = S.k;
    root.querySelectorAll('[data-act="k0"],[data-act="kmenos"]').forEach(b => { b.disabled = S.k === 0; });
    root.querySelectorAll('[data-act="kmas"],[data-act="kfin"]').forEach(b => { b.disabled = S.k === 25; });
    ajustarAltura();
  }

  function parar() { if (timer) { clearInterval(timer); timer = null; } S.play = false; }
  function irA(k) { parar(); S.k = Math.max(0, Math.min(25, k)); render(); }

  root.addEventListener("click", e => {
    const b = e.target.closest("[data-act]");
    if (!b || b.tagName === "SELECT" || b.tagName === "INPUT") return;
    const a = b.getAttribute("data-act"), v = b.getAttribute("data-v");
    if (a === "vista") { parar(); S.vista = v; S.k = 0; render(); }
    else if (a === "n") { S.n = Number(v); render(); }
    else if (a === "color") { S.color = v; render(); }
    else if (a === "presentar") { parar(); Object.assign(S, { vista: "presentacion", slide: 0, k: 0, n: 5, lado: "arriba", color: "niveles", esc: true }); render(); }
    else if (a === "salir") { parar(); S.vista = "general"; render(); }
    else if (a === "sant") { parar(); S.slide = Math.max(0, S.slide - 1); S.k = 0; render(); }
    else if (a === "ssig") { parar(); S.slide = Math.min(SLIDES.length - 1, S.slide + 1); S.k = 0; render(); }
    else if (a === "k0") irA(0);
    else if (a === "kmenos") irA(S.k - 1);
    else if (a === "kmas") irA(S.k + 1);
    else if (a === "kfin") irA(25);
    else if (a === "play") {
      if (S.play) { parar(); render(); return; }
      if (S.k >= 25) S.k = 0;
      S.play = true; render();
      timer = setInterval(() => {
        if (S.k >= 25) { parar(); render(); return; }
        S.k += 1;
        if (S.k >= 25) { parar(); render(); } else renderZona();
      }, 850);
    }
  });

  root.addEventListener("change", e => {
    const a = e.target.getAttribute("data-act");
    if (a === "lado") { S.lado = e.target.value; render(); }
    else if (a === "esc") { S.esc = e.target.checked; render(); }
    else if (a === "rango") { parar(); S.k = Number(e.target.value); render(); }
  });

  root.addEventListener("input", e => {
    if (e.target.getAttribute("data-act") === "rango") { if (S.play) parar(); S.k = Number(e.target.value); renderZona(); }
  });

  render();
})();
</script>
</body>
</html>
"""


def construir_html(datos: dict) -> str:
    """Inserta los datos calculados dentro de la página web."""
    datos_json = json.dumps(datos, ensure_ascii=False).replace("</", "<\\/")
    return PLANTILLA_HTML.replace("__DATOS__", datos_json)


# -----------------------------------------------------------------------------
# EJECUCIÓN
# -----------------------------------------------------------------------------
datos = precalcular()
pagina = construir_html(datos)

ARCHIVO_RESPALDO = "cuadro_magico_riesgo.html"
with open(ARCHIVO_RESPALDO, "w", encoding="utf-8") as archivo:
    archivo.write(pagina)

display(HTML(pagina))

if descargar_respaldo:
    try:
        from google.colab import files
        files.download(ARCHIVO_RESPALDO)
    except ImportError:
        print(f"Respaldo guardado en: {ARCHIVO_RESPALDO}")


In [ ]:
#@title ✔ Pruebas automáticas { display-mode: "form" }
#@markdown Corre primero la celda de la herramienta. Esta celda comprueba que todos los cálculos den lo que deben dar.

import unittest

if "cuadro_magico" not in globals():
    raise RuntimeError("Primero corre la celda «▶ Cuadro mágico vs. matriz de riesgo».")


class PruebasCuadroMagico(unittest.TestCase):

    def test_5x5_igual_al_ejemplo_conocido(self):
        esperado = [[17, 24, 1, 8, 15], [23, 5, 7, 14, 16], [4, 6, 13, 20, 22],
                    [10, 12, 19, 21, 3], [11, 18, 25, 2, 9]]
        self.assertEqual(cuadro_magico(5).tolist(), esperado)

    def test_todos_los_tamanos_y_lados_son_magicos(self):
        sumas = {3: 15, 5: 65, 7: 175, 9: 369}
        for n in TAMANOS:
            for lado in LADOS:
                with self.subTest(n=n, lado=lado):
                    v = verificar_magico(cuadro_magico(n, lado))
                    self.assertTrue(v["ok"])
                    self.assertEqual(v["suma_magica"], sumas[n])

    def test_rechaza_tamanos_invalidos(self):
        for n in (2, 4, 11, 0):
            with self.subTest(n=n):
                with self.assertRaises(ValueError):
                    cuadro_magico(n)

    def test_el_ultimo_numero_queda_en_el_lado_opuesto(self):
        M = cuadro_magico(5)
        self.assertEqual(M[0, 2], 1)    # centro de arriba
        self.assertEqual(M[4, 2], 25)   # centro de abajo


class PruebasMatrizRiesgo(unittest.TestCase):

    def test_maximo_y_conteo_de_niveles(self):
        R = matriz_riesgo()
        self.assertEqual(int(R.max()), 250)
        conteo = {}
        for v in R.flatten():
            conteo[nivel(int(v), 250)] = conteo.get(nivel(int(v), 250), 0) + 1
        self.assertEqual(conteo, {"Bajo": 6, "Mediano": 4, "Alto": 8, "Extremo": 7})

    def test_cuadro_de_prioridades(self):
        P, _ = matriz_prioridad(matriz_riesgo())
        esperado = [[25, 22, 17, 13, 8], [24, 19, 14, 9, 5], [23, 18, 12, 7, 3],
                    [21, 16, 11, 6, 2], [20, 15, 10, 4, 1]]
        self.assertEqual(P.tolist(), esperado)
        self.assertEqual(sorted(P.flatten().tolist()), list(range(1, 26)))

    def test_desequilibrio(self):
        P, _ = matriz_prioridad(matriz_riesgo())
        self.assertEqual(desequilibrio(cuadro_magico(5)), 0.0)
        self.assertEqual(desequilibrio(P), 19.8)


class PruebasInforme(unittest.TestCase):

    def test_totales_del_informe(self):
        r = prioridad_rav()
        self.assertEqual(r["total_riesgo"], 832)
        self.assertEqual(r["total_rav"], 1553.1)

    def test_orden_rav_igual_a_la_tabla_5(self):
        orden = [x["id"] for x in prioridad_rav()["filas"]]
        self.assertEqual(orden, ["R02", "R12", "R03", "R04", "R15", "R08", "R14", "R06",
                                 "R07", "R05", "R01", "R11", "R09", "R10", "R13"])
        primero = prioridad_rav()["filas"][0]
        self.assertEqual((primero["rav"], primero["pct"]), (640.0, 41.21))


resultado = unittest.main(argv=[""], exit=False, verbosity=2).result
print("\nTODO CORRECTO ✔" if resultado.wasSuccessful() else "\nHAY PRUEBAS FALLANDO ✘")


# Guía de estudio

Para ver el código: haz doble clic en la celda principal o usa el menú **Ver → Mostrar código**.
Está dividido en 4 capas, de arriba hacia abajo.

## Capa 1 · Datos del informe

Son constantes: `FRECUENCIA`, `IMPACTO`, `NIVELES`, `FACTORES_RAV` y `ESCENARIOS`. Salen tal cual del informe y nadie los cambia desde la interfaz.

## Capa 2 · Lógica (lo que más te van a preguntar)

| Función | Qué hace | Cómo explicarlo en una frase |
| --- | --- | --- |
| `cuadro_magico(n, lado)` | Genera el cuadro con el método siamés | "Pongo el 1 arriba al centro, avanzo en diagonal, si me salgo doy la vuelta con `%` y si está ocupado bajo una." |
| `verificar_magico(M)` | Suma filas, columnas y diagonales | "Compruebo que todo dé n(n²+1)/2 y que se usen todos los números del 1 al n²." |
| `pasos_siames()` | Repite el algoritmo guardando cada paso | "Es lo que alimenta la animación paso a paso." |
| `matriz_riesgo()` | Multiplica frecuencia × impacto | "Cada casilla es F × I; el máximo es 5 × 50 = 250." |
| `nivel(valor, maximo)` | Da bajo, mediano, alto o extremo | "Saco el % respecto a 250 y lo comparo con los umbrales 2, 5 y 20." |
| `matriz_prioridad(R)` | Numera las casillas del 1 al 25 | "Ordeno de mayor a menor riesgo; si empatan, gana la de mayor impacto." |
| `desequilibrio(M)` | Mide qué tan parejas son las sumas | "Es el promedio de cuánto se aleja cada suma del valor parejo. El mágico da 0 y el de prioridades 19,8." |
| `prioridad_rav()` | Aplica los factores RAV a los 15 escenarios | "Multiplico el riesgo por un factor según qué tan comprobado está." |

**Detalles de Python que conviene saber explicar:**

- `(fila - 1) % n`: el `%` (módulo) es lo que hace que la diagonal "dé la vuelta". Si `fila - 1` da -1, `-1 % 5` da 4: la última fila.
- `np.zeros((n, n), dtype=int)`: crea la cuadrícula vacía con ceros; un 0 significa "casilla libre".
- `M.sum(axis=1)` suma cada fila, `M.sum(axis=0)` suma cada columna y `np.trace(M)` suma la diagonal.
- `np.flipud`, `M.T` y `np.rot90`: reflejan o giran el cuadro para empezar en otro lado. Un cuadro mágico girado sigue siendo mágico.
- `sorted(..., key=lambda x: (-x[2], -impacto, -frecuencia))`: ordena por riesgo de mayor a menor (por eso el signo menos) y desempata por impacto.

## Capa 3 · Precálculo

`precalcular()` calcula todas las combinaciones (4 tamaños × 4 lados) y los 25 pasos de cada animación, y lo junta en un diccionario. Ese diccionario se convierte en JSON.

## Capa 4 · Interfaz

`construir_html()` mete el JSON dentro de una página web (HTML + CSS + JavaScript). La página **no calcula** riesgos ni cuadros: solo dibuja y cambia de pantalla. Por eso, si Colab se desconecta, los botones siguen funcionando.

## ¿Qué es RAV? (en simple)

La matriz solo mira qué tan seguido pasa algo y qué tan grave es. RAV agrega una pregunta: **¿qué tan comprobado está?**

- **Verificado** (hay evidencia): pesa más, hasta × 3,2.
- **Percibido** (alguien lo cree): se queda igual, × 1.
- **Filtrado** (ya tiene controles): casi desaparece.

Por eso R12 (cuentas de exempleados, con 9 cuentas comprobadas) sube del 4.º al 2.º lugar. Esta parte es opcional: sirve para conectar con el informe, pero el núcleo del trabajo es la comparación entre el cuadro mágico y el de prioridades.

## Guion del modo sustentación

Una idea por pantalla. No hace falta decirlo palabra por palabra.

1. **El reto.** El reto era generar un cuadro mágico con Python. Usé el método siamés: el 1 va en el centro de arriba, cada número sigue en diagonal y, si la casilla está ocupada, baja una. Así todas las filas, columnas y diagonales suman lo mismo.
2. **El cuadro mágico paso a paso.** Aquí se ve el algoritmo trabajando: cada número aplica una regla. Las sumas parciales van creciendo y, al terminar, todas dan 65.
3. **La matriz de riesgo.** En el informe cada riesgo se califica por qué tan seguido pasa y qué tan grave es. La casilla es frecuencia × impacto. El impacto crece más rápido (1, 3, 10, 25, 50), por eso lo grave pesa más. Aquí están mis 15 escenarios ubicados.
4. **Las prioridades paso a paso.** Uso los mismos números del 1 al 25, pero con otra regla: el más peligroso primero. La prioridad 1 es la casilla de 250. Si dos casillas valen lo mismo, gana la de mayor impacto.
5. **La comparación.** Los dos cuadros usan los números del 1 al 25. El cuadro mágico los reparte para que todo quede equilibrado, y la matriz de riesgo los ordena por peligro para saber qué atender primero. Por eso el mágico suma igual en todas partes y el de riesgo no.
6. **Cierre.** Llevado a mis 15 escenarios con el modelo RAV, la primera prioridad es la falla del aire acondicionado (41,21 %). Python calcula todo y verifica que el cuadro mágico sume bien; la interfaz solo muestra los resultados.

## Preguntas probables en la sustentación

**¿Qué relación tiene el cuadro mágico con la matriz de riesgo?**
Los dos son cuadrículas de 5×5 con los números del 1 al 25. Cambia la regla para acomodarlos: el mágico los reparte parejo y el de prioridades los ordena por peligro.

**¿Por qué el cuadro de prioridades no suma igual?**
Porque no debe: una matriz de riesgo tiene que concentrar la atención en lo grave. El desequilibrio de 19,8 lo demuestra.

**¿Cómo sabes que el código funciona?**
Corro la celda de pruebas: 9 pruebas comprueban el cuadro de 5×5 conocido, todos los tamaños y lados, los niveles, la tabla de prioridades y los totales del informe (832 y 1553,1).

**¿Por qué solo tamaños impares?**
El método siamés solo funciona con tamaños impares. Los pares necesitan otros métodos.

**¿Qué pasa con los empates?**
Gana la casilla de mayor impacto, porque en el informe el impacto pesa más que la frecuencia.

**¿La página web hace los cálculos?**
No. Python calcula y verifica todo; la página recibe los resultados ya listos y solo los muestra.
